In [ ]:
from crypto_utils import generate_key
generate_key()


In [ ]:
import pandas as pd

df = pd.read_csv("file.csv")
customers = pd.read_csv("customer.csv")
credit = pd.read_csv("creditCard.csv")

print("file.csv:", df.shape)
print("customer.csv:", customers.shape)
print("creditCard.csv:", credit.shape)


In [ ]:
print("file.csv columns:\n", df.columns.tolist())
print("\ncustomer.csv columns:\n", customers.columns.tolist())
print("\ncreditCard.csv columns:\n", credit.columns.tolist())

display(df.head(5))
display(customers.head(5))
display(credit.head(5))


In [ ]:
print("file.csv dtypes:\n", df.dtypes)
print("\ncustomer.csv dtypes:\n", customers.dtypes)
print("\ncreditCard.csv dtypes:\n", credit.dtypes)

print("\nMissing values (file.csv) - top 15:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nMissing values (customer.csv):")
print(customers.isna().sum())

print("\nMissing values (creditCard.csv):")
print(credit.isna().sum().sort_values(ascending=False))


In [ ]:
print("Unique customers in file.csv:", df["CustomerID"].nunique(dropna=True))
print("Unique customers in customer.csv:", customers["CustomerID"].nunique(dropna=True))

print("Unique transactions in file.csv:", df["Transaction_ID"].nunique(dropna=True))
print("Total rows in file.csv:", len(df))

# How many line items per transaction?
tx_lines = df.groupby("Transaction_ID").size()
print("\nLine items per transaction (describe):")
print(tx_lines.describe())

print("\nTop 10 transactions by number of rows:")
print(tx_lines.sort_values(ascending=False).head(10))


In [ ]:
tx_customer_counts = (
    df.dropna(subset=["Transaction_ID", "CustomerID"])
      .groupby("Transaction_ID")["CustomerID"]
      .nunique()
)

inconsistent = tx_customer_counts[tx_customer_counts > 1]
print("Transactions linked to >1 customer:", len(inconsistent))

# show examples
example_ids = inconsistent.head(5).index.tolist()
display(
    df[df["Transaction_ID"].isin(example_ids)]
      [["Transaction_ID", "CustomerID", "Transaction_Date", "Product_SKU", "Avg_Price"]]
      .sort_values(["Transaction_ID", "CustomerID"])
      .head(30)
)


In [ ]:
import pandas as pd

df = pd.read_csv("file.csv")

# 1) Find inconsistent Transaction_ID -> CustomerID mappings
tx_customer_nunique = (
    df.dropna(subset=["Transaction_ID", "CustomerID"])
      .groupby("Transaction_ID")["CustomerID"]
      .nunique()
)
inconsistent_ids = tx_customer_nunique[tx_customer_nunique > 1].index

print("Inconsistent Transaction_IDs (before cleaning):", len(inconsistent_ids))

# 2) Choose ONE CustomerID per inconsistent Transaction_ID:
        #use the mode (most frequent CustomerID)
        #tie-breaker: smallest CustomerID
sub = df.loc[df["Transaction_ID"].isin(inconsistent_ids), ["Transaction_ID", "CustomerID"]].dropna()
sub["CustomerID_int"] = sub["CustomerID"].astype(int)

def pick_mode_smallest(s):
    vc = s.value_counts()
    top = vc[vc == vc.max()].index
    return int(min(top))

mode_map = sub.groupby("Transaction_ID")["CustomerID_int"].agg(pick_mode_smallest)

# 3) Apply mapping
df_clean = df.copy()
mask_inconsistent = df_clean["Transaction_ID"].isin(inconsistent_ids)

before = df_clean.loc[mask_inconsistent, ["Transaction_ID", "CustomerID"]].copy()

df_clean.loc[mask_inconsistent, "CustomerID"] = (
    df_clean.loc[mask_inconsistent, "Transaction_ID"].map(mode_map).astype(float)
)

after = df_clean.loc[mask_inconsistent, ["Transaction_ID", "CustomerID"]].copy()

changed_rows = int((before["CustomerID"] != after["CustomerID"]).sum())

# 4) Verify integrity after cleaning
tx_customer_nunique_after = (
    df_clean.dropna(subset=["Transaction_ID", "CustomerID"])
            .groupby("Transaction_ID")["CustomerID"]
            .nunique()
)

print("Rows with CustomerID overwritten:", changed_rows)
print("Inconsistent Transaction_IDs (after cleaning):", int((tx_customer_nunique_after > 1).sum()))


In [ ]:
log_rows = before.copy()
log_rows["CustomerID_after"] = after["CustomerID"].values
log_rows["changed"] = (log_rows["CustomerID"] != log_rows["CustomerID_after"])

cleaning_log = (
    log_rows.groupby("Transaction_ID")
            .agg(
                distinct_customerids_before=("CustomerID", lambda s: s.dropna().nunique()),
                chosen_customerid_after=("CustomerID_after", lambda s: int(pd.Series(s.dropna()).mode().iloc[0])),
                rows_in_tx=("CustomerID", "size"),
                rows_changed=("changed", "sum"),
            )
            .sort_values(["rows_changed","rows_in_tx"], ascending=False)
)

cleaning_log.head(10)


In [ ]:
sku_desc = df_clean.groupby("Product_SKU")["Product_Description"].nunique(dropna=True)
sku_cat = df_clean.groupby("Product_SKU")["Product_Category"].nunique(dropna=True)

print("SKUs with >1 description:", (sku_desc > 1).sum())
print("SKUs with >1 category:", (sku_cat > 1).sum())


In [ ]:
file_ids = set(df_clean["CustomerID"].dropna().astype(int).unique())
cust_ids = set(customers["CustomerID"].astype(int).unique())

print("Customers in cleaned file.csv:", len(file_ids))
print("Customers in customer.csv:", len(cust_ids))
print("IDs in cleaned file.csv missing from customer.csv:", len(file_ids - cust_ids))
print("IDs in customer.csv missing from cleaned file.csv:", len(cust_ids - file_ids))

print("\nExample IDs in customer.csv but not in cleaned file.csv:")
print(sorted(list(cust_ids - file_ids))[:15])



In [ ]:
tx_customer_nunique_after = (
    df_clean.dropna(subset=["Transaction_ID", "CustomerID"])
            .groupby("Transaction_ID")["CustomerID"]
            .nunique()
)
assert (tx_customer_nunique_after > 1).sum() == 0
print("Check passed: each Transaction_ID maps to exactly one CustomerID.")


In [ ]:
import re

def norm_name_letters_only(s):
    if pd.isna(s):
        return None
    return re.sub(r"[^a-z]", "", str(s).lower())

cust_norm = customers["Name"].apply(norm_name_letters_only)
cc_norm = credit["Name"].apply(norm_name_letters_only)

mask = cc_norm.notna()
match_rate = (cust_norm[mask].values == cc_norm[mask].values).mean()

print("Name match rate after cleaning:", match_rate)

# Show a few before/after examples
sample = credit[mask].head(10).copy()
sample["customer_name"] = customers.loc[sample.index, "Name"].values
sample["credit_name"] = sample["Name"].values
sample["customer_clean"] = cust_norm.loc[sample.index].values
sample["credit_clean"] = cc_norm.loc[sample.index].values
display(sample[["customer_name","credit_name","customer_clean","credit_clean","CreditCard_provider"]])


In [ ]:
df = df_clean

In [ ]:
import sqlite3
import os

DB_PATH = "shop.db"

# Close old connection 
try:
    conn.close()
except:
    pass

# Delete old DB to avoid leftover tables created by to_sql
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE Customer (
    CustomerID      INTEGER PRIMARY KEY,
    Name            TEXT NOT NULL,
    Street          TEXT,
    City            TEXT,
    Postcode        TEXT,
    Gender          TEXT,
    Location        TEXT,
    Tenure_Months   INTEGER
);

CREATE TABLE Product (
    Product_SKU          TEXT PRIMARY KEY,
    Product_Description  TEXT NOT NULL,
    Product_Category     TEXT
);

CREATE TABLE SalesTransaction (
    Transaction_ID     INTEGER PRIMARY KEY,
    CustomerID         INTEGER NOT NULL,
    Transaction_Date   TEXT,
    Month              INTEGER,
    Offline_Spend      REAL,
    Online_Spend       REAL,
    Delivery_Charges   REAL,
    Coupon_Status      TEXT,
    Coupon_Code        TEXT,
    Discount_pct       REAL,
    FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID)
);

CREATE TABLE TransactionItem (
    Transaction_ID   INTEGER NOT NULL,
    Product_SKU      TEXT NOT NULL,
    Quantity         REAL,
    Avg_Price        REAL,
    GST              REAL,
    PRIMARY KEY (Transaction_ID, Product_SKU),
    FOREIGN KEY (Transaction_ID) REFERENCES SalesTransaction(Transaction_ID),
    FOREIGN KEY (Product_SKU) REFERENCES Product(Product_SKU)
);

CREATE TABLE CreditCard (
    CustomerID            INTEGER PRIMARY KEY,
    CreditCard_number     TEXT,
    CreditCard_provider   TEXT,
    FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID)
);
""")
conn.commit()

print("Fresh DB created with constraints.")


In [ ]:
cur.execute("""
SELECT name FROM sqlite_master
WHERE type='table';
""")

tables = cur.fetchall()
tables


In [ ]:
for name in ["customer_df", "product_df", "transaction_df", "transaction_item_df", "creditcard_df"]:
    print(f"{name} exists:", name in globals())


In [ ]:
import pandas as pd
import re

df = pd.read_csv("file.csv")
customers = pd.read_csv("customer.csv")
credit = pd.read_csv("creditCard.csv")

#Fix Transaction_ID -> CustomerID inconsistencies ---
tx_customer_nunique = (
    df.dropna(subset=["Transaction_ID", "CustomerID"])
      .groupby("Transaction_ID")["CustomerID"]
      .nunique()
)
inconsistent_ids = tx_customer_nunique[tx_customer_nunique > 1].index

sub = df.loc[df["Transaction_ID"].isin(inconsistent_ids), ["Transaction_ID", "CustomerID"]].dropna()
sub["CustomerID_int"] = sub["CustomerID"].astype(int)

def pick_mode_smallest(s):
    vc = s.value_counts()
    top = vc[vc == vc.max()].index
    return int(min(top))

mode_map = sub.groupby("Transaction_ID")["CustomerID_int"].agg(pick_mode_smallest)

df_clean = df.copy()
mask_inconsistent = df_clean["Transaction_ID"].isin(inconsistent_ids)
df_clean.loc[mask_inconsistent, "CustomerID"] = (
    df_clean.loc[mask_inconsistent, "Transaction_ID"].map(mode_map).astype(float)
)

#Drop junk index column
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

df_clean["Transaction_Date"] = pd.to_datetime(df_clean["Transaction_Date"], errors="coerce")

#Product table
product_df = (
    df_clean[["Product_SKU", "Product_Description", "Product_Category"]]
    .dropna(subset=["Product_SKU"])
    .drop_duplicates(subset=["Product_SKU"])
    .reset_index(drop=True)
)

#Transaction table
tx_cols = [
    "Transaction_ID", "CustomerID", "Transaction_Date", "Month",
    "Offline_Spend", "Online_Spend", "Delivery_Charges",
    "Coupon_Status", "Coupon_Code", "Discount_pct"
]
transaction_df = (
    df_clean[tx_cols]
    .dropna(subset=["Transaction_ID"])
    .groupby("Transaction_ID", as_index=False)
    .agg({
        "CustomerID": "first",
        "Transaction_Date": "first",
        "Month": "first",
        "Offline_Spend": "first",
        "Online_Spend": "first",
        "Delivery_Charges": "first",
        "Coupon_Status": "first",
        "Coupon_Code": "first",
        "Discount_pct": "first",
    })
)

transaction_df["Transaction_ID"] = transaction_df["Transaction_ID"].astype(int)
transaction_df["CustomerID"] = transaction_df["CustomerID"].astype(int)

# Store Transaction_Date as ISO text for SQLite
transaction_df["Transaction_Date"] = transaction_df["Transaction_Date"].dt.strftime("%Y-%m-%d")

#TransactionItem table
item_cols = ["Transaction_ID", "Product_SKU", "Quantity", "Avg_Price", "GST"]
transaction_item_df = (
    df_clean[item_cols]
    .dropna(subset=["Transaction_ID", "Product_SKU"])
    .copy()
)
transaction_item_df["Transaction_ID"] = transaction_item_df["Transaction_ID"].astype(int)

transaction_item_df = (
    transaction_item_df
    .groupby(["Transaction_ID", "Product_SKU"], as_index=False)
    .agg({"Quantity": "sum", "Avg_Price": "mean", "GST": "mean"})
)

# Customer table
demo = (
    df_clean[["CustomerID", "Gender", "Location", "Tenure_Months"]]
    .dropna(subset=["CustomerID"])
    .copy()
)
demo["CustomerID"] = demo["CustomerID"].astype(int)
demo = demo.groupby("CustomerID", as_index=False).agg({
    "Gender": "first",
    "Location": "first",
    "Tenure_Months": "first",
})

customer_df = customers.merge(demo, on="CustomerID", how="left")

#CreditCard table (map by cleaned name)
def norm_name_letters_only(s):
    if pd.isna(s):
        return None
    return re.sub(r"[^a-z]", "", str(s).lower())

name_to_id = (
    pd.DataFrame({
        "CustomerID": customer_df["CustomerID"],
        "name_clean": customer_df["Name"].apply(norm_name_letters_only)
    })
    .dropna()
    .drop_duplicates(subset=["name_clean"])
    .set_index("name_clean")["CustomerID"]
)

credit_df = credit.copy()
credit_df["name_clean"] = credit_df["Name"].apply(norm_name_letters_only)
credit_df["CustomerID"] = credit_df["name_clean"].map(name_to_id)

#Store credit card number as TEXT (avoid float/scientific notation)
credit_df["CreditCard_number"] = credit_df["CreditCard_number"].apply(
    lambda x: None if pd.isna(x) else str(int(float(x)))
)

creditcard_df = (
    credit_df[["CustomerID", "CreditCard_number", "CreditCard_provider"]]
    .dropna(subset=["CustomerID"])
    .drop_duplicates(subset=["CustomerID"])
    .copy()
)
creditcard_df["CustomerID"] = creditcard_df["CustomerID"].astype(int)

print("DataFrames ready:",
      customer_df.shape, product_df.shape, transaction_df.shape, transaction_item_df.shape, creditcard_df.shape)


In [ ]:
for name in ["customer_df", "product_df", "transaction_df", "transaction_item_df", "creditcard_df"]:
    print(f"{name} exists:", name in globals())


In [ ]:
customer_df.to_sql("Customer", conn, if_exists="append", index=False)
product_df.to_sql("Product", conn, if_exists="append", index=False)
transaction_df.to_sql("SalesTransaction", conn, if_exists="append", index=False)
transaction_item_df.to_sql("TransactionItem", conn, if_exists="append", index=False)

from crypto_utils import generate_key, encrypt_text
generate_key()
creditcard_df["CreditCard_number"] = creditcard_df["CreditCard_number"].apply(encrypt_text)

creditcard_df.to_sql("CreditCard", conn, if_exists="append", index=False)

conn.commit()
print("Data inserted using append (constraints preserved).")


In [ ]:
cur.execute("PRAGMA table_info(CreditCard);")
cur.fetchall()


In [ ]:
cur.execute("""
SELECT st.Transaction_ID, st.CustomerID, c.Name, ti.Product_SKU, p.Product_Description, ti.Quantity, ti.Avg_Price
FROM SalesTransaction st
JOIN Customer c ON c.CustomerID = st.CustomerID
JOIN TransactionItem ti ON ti.Transaction_ID = st.Transaction_ID
JOIN Product p ON p.Product_SKU = ti.Product_SKU
LIMIT 10;
""")
cur.fetchall()
